In [ ]:
# !pip install google-genai pip requests

In [18]:
# !pip install -r requirements.txt


In [1]:
from google import genai
from google.genai import errors
import requests
from google.genai import types

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

GEMINI_API_KEY=os.getenv("GEMINI_API_KEY")
FLIGHT_SEARCH_API_key=os.getenv("FLIGHT_SEARCH_API_key")

In [3]:
client = genai.Client()

print("Available models:")
print("-" * 30)

for model in client.models.list():
    # print(model.model_dump())
    # In the new SDK, check if 'generate_content' is in supported_actions
    if "generateContent" in model.supported_actions:
        if "gemini-3" in model.name:
            print(model.name)

Available models:
------------------------------
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/gemini-3.1-flash-tts-preview


In [ ]:
import pandas as pd

df = pd.read_csv('airports_list.dat', header=None
            ,names=['id', 'name', 'city', 'country', 'iata', 'icao', 'latitude', 'longitude', 'altitude', 'timezone', 'dst','tz_database_time_zone', 'type', 'source'])

airport_code_df = df[["name", "city", "country", "iata"]]
# airport_code_df.head()
airport_code_df[airport_code_df['country'] == 'India']

,iata,airport_name,city,country
2836,AMD,Sardar Vallabhbhai Patel International Airport,Ahmedabad,India
2837,AKD,Akola Airport,Akola,India
2838,IXU,Aurangabad Airport,Aurangabad,India
2839,BOM,Chhatrapati Shivaji International Airport,Mumbai,India
2840,PAB,Bilaspur Airport,Bilaspur,India
...,...,...,...,...
7555,SAG,Shirdi Airport,Shirdi,India
7556,PYB,Jeypore Airport,Jeypore,India
7644,KQH,Kishangarh Airport,Ajmer,India
7645,CNN,Kannur International Airport,Kannur,India


In [10]:
import os
import re
import json
import requests
import pandas as pd
from functools import lru_cache

def _load_airports() -> pd.DataFrame:
    """
    Load airport data from the CSV specified in AIRPORTS_DATA_PATH env var.
    Falls back to downloading OpenFlights public dataset if file not found.

    Expected columns (flexible — we remap common variants):
        name, city, country, iata
    """
    df = pd.read_csv('airports_list.dat', header=None
            ,names=['id', 'name', 'city', 'country', 'iata', 'icao', 'latitude', 'longitude', 'altitude', 'timezone', 'dst','tz_database_time_zone', 'type', 'source'])

    airport_code_df = df[["name", "city", "country", "iata"]].copy()

    # df = df[["name", "city", "country", "iata"]].copy()

    airport_code_df = airport_code_df.fillna("")
    return airport_code_df.reset_index(drop=True)



def _normalize(text: str) -> str:
    """Lowercase, strip extra spaces, remove punctuation for fuzzy matching."""
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)   # remove punctuation
    text = re.sub(r"\s+", " ", text)       # collapse whitespace
    return text



In [11]:
def lookup_airport_code(city_name: str) -> str:
    """
    Find the IATA airport code for a given city name.
    Handles variations in case, spacing, and common alternate spellings.

    Args:
        city_name: The city or airport name to look up (e.g. "Delhi", "new york", "mumbai")

    Returns:
        JSON with matched airports including IATA code, city, country.
        Returns top match first. If multiple matches exist, all are returned
        so the LLM can pick the most appropriate one.
    """
    df = _load_airports()
    query = _normalize(city_name)

    # Build normalized lookup columns (computed once, cached via the df reference)
    if "_city_norm" not in df.columns:
        df["_city_norm"] = df["city"].apply(_normalize)
        df["_name_norm"] = df["name"].apply(_normalize)

    # Priority 1: Exact city match
    exact = df[df["_city_norm"] == query]

    # Priority 2: City starts with query
    starts = df[df["_city_norm"].str.startswith(query)] if exact.empty else pd.DataFrame()

    # Priority 3: City contains query anywhere
    contains_city = (
        df[df["_city_norm"].str.contains(query, regex=False)]
        if exact.empty and starts.empty
        else pd.DataFrame()
    )

    # Priority 4: Airport name contains query
    contains_name = (
        df[df["_name_norm"].str.contains(query, regex=False)]
        if exact.empty and starts.empty and contains_city.empty
        else pd.DataFrame()
    )

    # Merge in priority order, deduplicate
    candidates = pd.concat([exact, starts, contains_city, contains_name])
    candidates = candidates.drop_duplicates(subset=["iata"]).head(5)

    if candidates.empty:
        return json.dumps({
            "status": "not_found",
            "message": f"No airport found matching '{city_name}'. "
                       "Please try the full city name or an alternate spelling.",
            "matches": []
        })

    matches = candidates[["name", "city", "country", "iata"]].to_dict(orient="records")
    return json.dumps({
        "status": "found",
        "query": city_name,
        "matches": matches,
        "best_match": matches[0]   # highest-priority hit
    }, indent=2)


In [14]:
lookup_airport_code(city_name = "jaipur")


'{\n  "status": "found",\n  "query": "jaipur",\n  "matches": [\n    {\n      "name": "Jaipur International Airport",\n      "city": "Jaipur",\n      "country": "India",\n      "iata": "JAI"\n    }\n  ],\n  "best_match": {\n    "name": "Jaipur International Airport",\n    "city": "Jaipur",\n    "country": "India",\n    "iata": "JAI"\n  }\n}'

In [ ]:
import requests
import json

SEARCHAPI_KEY = FLIGHT_SEARCH_API_key

def search_robust_flights(departure_id: str, arrival_id: str, outbound_date: str) -> dict:
    url = "https://www.searchapi.io/api/v1/search"
    
    params = {
        "engine": "google_flights",
        "departure_id": departure_id.upper().strip(),
        "arrival_id": arrival_id.upper().strip(),
        "outbound_date": outbound_date.strip(),
        "flight_type": "one_way", 
        "currency": "INR",          
        "hl": "en",                 
        "gl": "IN",                 
        "api_key": SEARCHAPI_KEY
    }
    
    try:
        print(f"📡 Querying Google Flights for {params['departure_id']} ➡️ {params['arrival_id']} on {params['outbound_date']}...\n")
        response = requests.get(url, params=params)
        
        if response.status_code != 200:
            return {"status": "error", "message": f"API HTTP Error {response.status_code}"}
            
        data = response.json()
        raw_flights = data.get("best_flights") or data.get("other_flights") or []
        
        if not raw_flights:
            return {"status": "success", "flights": []}
            
        final_selections = []
        
        # INCREASED LIMIT: We pull up to 15 flights from the API so we can sort them later
        for option in raw_flights[:15]:
            segments = option.get("flights", [])
            if not segments:
                continue
                
            num_stops = len(segments) - 1
            stops_text = "Non-stop" if num_stops == 0 else f"{num_stops} Stop(s)"
            primary_airline = segments[0].get("airline", "Unknown Airline")
            
            dep_info = segments[0].get("departure_airport", {})
            departure_time = dep_info.get("time", "N/A")
            
            arr_info = segments[-1].get("arrival_airport", {})
            arrival_time = arr_info.get("time", "N/A")
            
            total_minutes = option.get("total_duration") or segments[0].get("duration") or 0
            
            flight_payload = {
                "airline": primary_airline,
                "price_raw": int(option.get('price', 0)), # Keep integer for sorting
                "duration_mins": total_minutes,            # Keep integer for sorting
                "duration_str": f"{total_minutes // 60}h {total_minutes % 60}m",
                "stops": stops_text,
                "departure_time": departure_time,
                "arrival_time": arrival_time
            }
            final_selections.append(flight_payload)
            
        return {"status": "success", "flights": final_selections}
        
    except Exception as e:
        return {"status": "error", "message": str(e)}

def print_flight_summary(flight_data: dict):
    """Sorts and prints the flight data into a human-readable agent summary."""
    if flight_data.get("status") != "success" or not flight_data.get("flights"):
        print("Could not retrieve flights or an error occurred.")
        return

    flights = flight_data["flights"]

    # 1. Sort by Price (Lowest to Highest)
    cheapest_flights = sorted(flights, key=lambda x: x["price_raw"])[:5]

    # 2. Sort by Duration / Time (Fastest to Slowest)
    fastest_flights = sorted(flights, key=lambda x: x["duration_mins"])[:5]

    print("=========================================================================")
    print(" ✈️  FLIGHT OPTIONS SUMMARY GENERATED BY YOUR AGENT  ✈️ ")
    print("=========================================================================\n")

    print("💵 HERE ARE THE PRICE-WISE FLIGHT OPTIONS (LOWEST PRICE):")
    print("-" * 73)
    for i, f in enumerate(cheapest_flights, 1):
        print(f"{i}. {f['airline']:<15} | Price: ₹{f['price_raw']:<6} | Duration: {f['duration_str']:<7} | {f['stops']:<9} | Dep: {f['departure_time']} -> Arr: {f['arrival_time']}")

    print("\n")

    print("⏱️  HERE ARE THE TIME-WISE FLIGHT OPTIONS (SHORTEST DURATION):")
    print("-" * 73)
    for i, f in enumerate(fastest_flights, 1):
        print(f"{i}. {f['airline']:<15} | Price: ₹{f['price_raw']:<6} | Duration: {f['duration_str']:<7} | {f['stops']:<9} | Dep: {f['departure_time']} -> Arr: {f['arrival_time']}")
    print("=========================================================================")


In [45]:
# --- Run Code ---
raw_result = search_robust_flights(
    departure_id="DEL",
    arrival_id="BOM",
    outbound_date="2026-07-20"
)

# Call our new printer function
print_flight_summary(raw_result)

📡 Querying Google Flights for DEL ➡️ BOM on 2026-07-20...

 ✈️  FLIGHT OPTIONS SUMMARY GENERATED BY YOUR AGENT  ✈️ 

💵 HERE ARE THE PRICE-WISE FLIGHT OPTIONS (LOWEST PRICE):
-------------------------------------------------------------------------
1. IndiGo          | Price: ₹5928   | Duration: 5h 5m   | 1 Stop(s) | Dep: 21:50 -> Arr: 02:55
2. Akasa Air       | Price: ₹6964   | Duration: 2h 25m  | Non-stop  | Dep: 08:40 -> Arr: 11:05
3. Akasa Air       | Price: ₹6964   | Duration: 2h 20m  | Non-stop  | Dep: 10:00 -> Arr: 12:20


⏱️  HERE ARE THE TIME-WISE FLIGHT OPTIONS (SHORTEST DURATION):
-------------------------------------------------------------------------
1. Akasa Air       | Price: ₹6964   | Duration: 2h 20m  | Non-stop  | Dep: 10:00 -> Arr: 12:20
2. Akasa Air       | Price: ₹6964   | Duration: 2h 25m  | Non-stop  | Dep: 08:40 -> Arr: 11:05
3. IndiGo          | Price: ₹5928   | Duration: 5h 5m   | 1 Stop(s) | Dep: 21:50 -> Arr: 02:55


In [22]:
client = genai.Client(api_key=GEMINI_API_KEY)


response = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents='Explain hallucinations in large language models and how to mitigate them. give very concise answer',
        config={
        'max_output_tokens': 100,
        'temperature': 0.7}
    )

print(response.text)

### What are Hallucinations?
Hallucinations occur when an LLM generates text that is grammatically correct and fluent but factually incorrect, nonsensical, or unfaithful to the provided source material. Because LLMs are probabilistic "next-token predictors" rather than databases of facts, they prioritize pattern matching over truth.

### How to Mitigate Them
1.  **Retrieval-Augmented Generation (RAG):** Connect the model to trusted, external
